# LACAN Protected Atoms & Bonds — Feature Showcase

Demonstrates:
- **Protected atoms**: skipped by all mutation and replacement operations
- **Protected bonds**: ignored by the LACAN scorer (for molecules with forced-bad motifs)
- **Scaffold decoration, ring replacement, linker replacement, substituent replacement**
- **mol_cleaner**: iteratively fix a failing molecule while preserving its good parts

In [ ]:
from rdkit import Chem
from rdkit.Chem.Draw import MolsToGridImage
from lacan import lacan
from lacan.protect import (
    protect_atoms_for_idx,
    protect_atoms_matching_smarts,
    unprotect_atoms_for_idx,
    unprotect_atoms_all,
    protect_bonds_for_idx,
    protect_rejected_bonds,
    get_protected_atom_indices,
    get_protected_bond_indices,
    score_mol_ignoring_protected_bonds,
    mol_cleaner,
)
from lacan.replace import decorate_scaffold, replace_substituent, replace_ring, replace_linker
from lacan.mutate import apply_mutations

profile = lacan.load_profile("chembl")
print("Profile loaded.")

## 1. Basic protection API

Protect specific atoms by index or by SMARTS pattern.

In [ ]:
fluoxetine = Chem.MolFromSmiles("CNCCC(c1ccccc1)Oc1ccc(C(F)(F)F)cc1")

# Protect the CF3-phenyl ring by SMARTS
mol = protect_atoms_matching_smarts(fluoxetine, "c1ccc(C(F)(F)F)cc1")
protected = get_protected_atom_indices(mol)
print(f"Protected atom indices: {protected}")
print(f"Atom symbols: {[mol.GetAtomWithIdx(i).GetSymbol() for i in protected]}")

In [ ]:
# Protect by index, then selectively unprotect
mol2 = protect_atoms_for_idx(fluoxetine, [0, 1, 2, 3])
print("Protected:", get_protected_atom_indices(mol2))
mol2 = unprotect_atoms_for_idx(mol2, [0, 1])
print("After unprotect [0,1]:", get_protected_atom_indices(mol2))
mol2 = unprotect_atoms_all(mol2)
print("After unprotect_all:", get_protected_atom_indices(mol2))

## 2. Protected bonds — scoring with known-bad motifs

Aspirin scores 0 because it contains a structural alert filtered from the ChEMBL training set.
If you need to work with it anyway, protect the offending bonds so they are excluded from scoring.

In [ ]:
aspirin = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")

score_raw, info = lacan.score_mol(aspirin, profile)
print(f"Aspirin raw score:      {score_raw:.3f}")
print(f"Bad bond indices:       {info['bad_bonds']}")

# Protect the rejected bonds so they are excluded from scoring
aspirin_prot = protect_rejected_bonds(aspirin, profile)
print(f"\nProtected bond indices: {get_protected_bond_indices(aspirin_prot)}")

score_prot, info2 = score_mol_ignoring_protected_bonds(aspirin_prot, profile)
print(f"Score (bonds protected): {score_prot:.3f}")
print(f"Remaining bad bonds:     {info2['bad_bonds']}")

## 3. Protected atoms block mutations

Any mutation reaction that would touch a protected atom is skipped.

In [ ]:
toluene = Chem.MolFromSmiles("Cc1ccccc1")

result_free = apply_mutations(toluene, profile, score_threshold=0.0)
print(f"Mutations on free toluene:    {len(result_free)} products")

# Protect the ring — only the methyl should be mutable now
toluene_prot = protect_atoms_matching_smarts(toluene, "c1ccccc1")
result_prot = apply_mutations(toluene_prot, profile, score_threshold=0.0)
print(f"Mutations with ring protected: {len(result_prot)} products")

phenyl = Chem.MolFromSmarts("c1ccccc1")
all_intact = all(m.HasSubstructMatch(phenyl) for m in result_prot)
print(f"All products preserved phenyl: {all_intact}")

## 4. Scaffold decoration

Add substituents to a molecule. Protected atoms are skipped.

- **Hydrogen mode**: replace H atoms with fragments from the corpus
- **Dummy mode**: fill pre-placed `*` attachment points

In [ ]:
ibuprofen = Chem.MolFromSmiles("CC(C)Cc1ccc(cc1)C(C)C(=O)O")

# Decorate freely
decorated = decorate_scaffold(ibuprofen, profile, score_threshold=0.5,
                              mode="Hydrogen", replacements_per_mol=1, n_replacements=30)
print(f"Decorated (free):           {len(decorated)} products")

# Protect the ring — only aliphatic chain gets decorated
mol_prot = protect_atoms_matching_smarts(ibuprofen, "c1ccccc1")
decorated_prot = decorate_scaffold(mol_prot, profile, score_threshold=0.5,
                                   mode="Hydrogen", replacements_per_mol=1, n_replacements=30)
print(f"Decorated (ring protected): {len(decorated_prot)} products")

if decorated:
    MolsToGridImage(decorated[:8], molsPerRow=4, subImgSize=(300, 200))

In [ ]:
# Dummy mode: pre-place * attachment points on a scaffold
scaffold = Chem.MolFromSmiles("c1c(*)c(*)co1")
result = decorate_scaffold(scaffold, profile, score_threshold=0.3,
                           mode="Dummy", n_replacements=20)
print(f"Decorated scaffold (Dummy mode): {len(result)} products")
if result:
    MolsToGridImage(result[:8], molsPerRow=4, subImgSize=(300, 200))

## 5. Scaffold (ring) replacement

Replace a ring system with a different ring from the fragment corpus.
Protected rings are not replaced.

In [ ]:
result = replace_ring(ibuprofen, profile, score_threshold=0.5, n_replacements=50)
print(f"Ring replacements: {len(result)} products")
for m in result[:5]:
    print(" ", Chem.MolToSmiles(m))
if result:
    MolsToGridImage(result[:8], molsPerRow=4, subImgSize=(300, 200))

In [ ]:
# With ring protected — replace_ring should skip it
mol_prot = protect_atoms_matching_smarts(ibuprofen, "c1ccccc1")
result_prot = replace_ring(mol_prot, profile, score_threshold=0.0, n_replacements=30)
print(f"Ring replacements (protected):   {len(result_prot)} products  <- should be 0")

mol_free = unprotect_atoms_all(mol_prot)
result_free = replace_ring(mol_free, profile, score_threshold=0.0, n_replacements=30)
print(f"Ring replacements (unprotected): {len(result_free)} products")

## 6. Substituent replacement

Swap a non-ring group with a different substituent from the corpus.

In [ ]:
result = replace_substituent(ibuprofen, profile, score_threshold=0.5, n_replacements=50)
print(f"Substituent replacements: {len(result)} products")
if result:
    MolsToGridImage(result[:8], molsPerRow=4, subImgSize=(300, 200))

## 7. Linker replacement (new)

Replace the chain or atom bridging two ring systems.
Protected linkers are not replaced.

In [ ]:
diphenylmethane = Chem.MolFromSmiles("c1ccc(Cc2ccccc2)cc1")

result = replace_linker(diphenylmethane, profile, score_threshold=0.3, n_replacements=50)
print(f"Linker replacements: {len(result)} products")
for m in result[:5]:
    print(" ", Chem.MolToSmiles(m))
if result:
    MolsToGridImage(result[:8], molsPerRow=4, subImgSize=(300, 200))

In [ ]:
# Protect the CH2 linker — nothing should be replaced
dpm_prot = protect_atoms_matching_smarts(diphenylmethane, "[CH2]")
result_prot = replace_linker(dpm_prot, profile, score_threshold=0.0, n_replacements=20)
print(f"Linker replacements (protected): {len(result_prot)} products  <- should be 0")

## 8. mol_cleaner — iterative fixing with lateral moves

`mol_cleaner` uses a greedy best-first search over single mutations to eliminate
all LACAN bond violations. The key challenge is **molecules with multiple
violations**: a single mutation can rarely fix two problems at once, so the
cleaner must be able to make *lateral moves* — accepting a mutation that keeps
the same violation count but reorganises the bad bonds — to unlock a subsequent
improvement elsewhere.

**Algorithm per iteration:**
1. Generate all single-step mutations
2. Accept the candidate with the **fewest violations** (improvement step)
3. If no mutation reduces violations, accept the **highest-scoring** candidate
   with the same violation count (lateral move) — up to `lateral_patience` times
4. Re-protect newly-clean bonds after each step, so only bad regions remain mutable

This means a molecule like `CC(=O)Oc1ccccc1C(=O)OF` (3 violations) can reach
0 violations via a sequence: fix violation A → lateral move to reposition violation B
→ fix violation B → fix violation C, rather than needing a single mutation that
fixes all three simultaneously.

In [ ]:
aspirin = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
score_raw, info = lacan.score_mol(aspirin, profile)
print(f"Aspirin raw score:  {score_raw:.3f}")
print(f"Bad bond indices:   {info['bad_bonds']}")
print(f"Num violations:     {len(info['bad_bonds'])}")

cleaned = mol_cleaner(aspirin, profile, score_threshold=0.5, max_iter=100, lateral_patience=5)
if cleaned:
    print(f"\nCleaned SMILES:  {Chem.MolToSmiles(cleaned)}")
    print(f"Cleaned score:   {lacan.score_mol(cleaned, profile)[0]:.3f}")
    d = MolsToGridImage([aspirin, cleaned], legends=["original (score=0)", "cleaned"])
    display(d)
else:
    print("\nCould not clean — try increasing max_iter or lateral_patience")

In [ ]:
# Demonstrate with a molecule that has two independent violations
# CC(=O)Oc1ccccc1C(=O)OF — aspirin + fluorine on carboxyl (two bad regions)
mol2 = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)OF")
if mol2:
    score2, info2 = lacan.score_mol(mol2, profile)
    print(f"Two-violation mol score: {score2:.3f}, violations: {len(info2['bad_bonds'])}")
    cleaned2 = mol_cleaner(mol2, profile, score_threshold=0.5, max_iter=150, lateral_patience=8)
    if cleaned2:
        print(f"Cleaned: {Chem.MolToSmiles(cleaned2)}")
        print(f"Score:   {lacan.score_mol(cleaned2, profile)[0]:.3f}")
    else:
        print("Could not clean within budget")

## 9. Summary of the protection API

| Function | Purpose |
|---|---|
| `protect_atoms_for_idx(mol, [i,j,...])` | Protect specific atoms by index |
| `protect_atoms_matching_smarts(mol, smarts)` | Protect atoms matching a SMARTS pattern |
| `unprotect_atoms_for_idx(mol, [i,j,...])` | Remove protection from specific atoms |
| `unprotect_atoms_all(mol)` | Remove all atom protection |
| `protect_bonds_for_idx(mol, [i,j,...])` | Protect specific bonds by bond index |
| `protect_rejected_bonds(mol, profile)` | Auto-protect failing bonds |
| `score_mol_ignoring_protected_bonds(mol, profile)` | Score, skipping protected bonds |
| `mol_cleaner(mol, profile)` | Iteratively fix bad regions with lateral moves |

**Key rules:**
- Protected atoms are never modified by any operation
- Protected bonds are excluded from scoring
- All functions return new molecules — originals are never modified
- `mol_cleaner` re-protects after each improvement, so only the remaining bad regions stay mutable